<a href="https://colab.research.google.com/github/Al-Amin95/PromptInjectionDetectionSystem/blob/model-train-comparison/notebiooks/03_distilbert_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#DistilBERT Fine Tuning

# Part 1 — Fine-Tuning Concept and Objective

## 1.1 What Fine-Tuning Means

Fine-tuning means taking a model that has already learned general language patterns and training it further on a specific task.

In this project, the specific task is prompt injection detection. The model will read a prompt and classify it as either SAFE or INJECTION.

The model is not being trained to answer the prompt. It is being trained to detect whether the prompt is safe or potentially malicious.

## 1.2 Why DistilBERT Is Used First

DistilBERT is used first because it is the baseline model for this project.

A baseline model is the first model trained in an experiment. Its results provide a starting point for comparison. After DistilBERT is trained and evaluated, the same general process will be used for RoBERTa and SecBERT.

The final model will not be selected until all three models have been trained and compared using evaluation metrics.

## 1.3 Task Type

The task is binary text classification.

Binary means there are two possible classes.

The two classes are:

- 0 = SAFE
- 1 = INJECTION

SAFE means the prompt is treated as a normal user request.

INJECTION means the prompt contains prompt-injection or adversarial instruction behaviour.

## 1.4 Dataset Used

This notebook uses the prepared deepset prompt-injection dataset.

The prepared files used later in this notebook are:

- clean_train.parquet
- clean_validation.parquet
- clean_test.parquet

The model input column is:

- text_for_model

The target label column is:

- label

## 1.5 Notebook Objective

The objective of this notebook is to fine-tune DistilBERT for binary prompt injection detection.

The trained model should take a prompt as input and return one of two predictions:

- SAFE
- INJECTION

This notebook will train DistilBERT, evaluate it on validation and test data, save metrics, save predictions, analyse false positives and false negatives, and prepare the result for comparison with RoBERTa and SecBERT.

## 1.6 Expected Outcome of This Notebook

By the end of this notebook, the following outputs should be produced:

- Fine-tuned DistilBERT model
- Saved tokenizer
- Validation metrics
- Test metrics
- Confusion matrix
- Validation prediction file
- Test prediction file
- False positive examples
- False negative examples
- Training summary

DistilBERT will then be ready for comparison with RoBERTa and SecBERT.

# Part 2 — Notebook Setup and Repository Update

In [ ]:
# 1 mount google drive
from google.colab import drive
drive.mount("/content/drive")

print("google drive mounted")


# 2 install libraries
!pip install -q transformers datasets accelerate evaluate scikit-learn pandas numpy matplotlib pyarrow

print("libraries installed")


# 3 import python packages
from pathlib import Path
import os
import sys
import time
import json
import shutil
import random
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import transformers
import datasets
import sklearn

print("packages imported")


# 4 define github repo
github_username = "Al-Amin95"
repo_name = "PromptInjectionDetectionSystem"
repo_url = f"https://github.com/{github_username}/{repo_name}.git"

branch_name = "model-train-comaprison"

repo_parent_dir = Path("/content")
project_root = repo_parent_dir / repo_name

print("repo url:", repo_url)
print("branch:", branch_name)
print("project root:", project_root)


# 5 clone or update repo
if project_root.exists():
    print("repo already exists, updating")
    %cd {project_root}
    !git fetch origin
else:
    print("repo not found, cloning")
    %cd {repo_parent_dir}
    !git clone {repo_url}
    %cd {project_root}
    !git fetch origin


# 6 checkout working branch
print("checking branch")

!git checkout {branch_name} || git checkout -b {branch_name} origin/{branch_name}

print("pulling latest changes")
!git pull origin {branch_name}


# 7 check repo status
print("current folder")
!pwd

print("current branch")
!git branch --show-current

print("git status")
!git status --short


# 8 define project paths
raw_data_dir = project_root / "data" / "raw"
processed_data_dir = project_root / "data" / "processed"
notebooks_dir = project_root / "notebooks"
results_tables_dir = project_root / "results" / "tables"
results_figures_dir = project_root / "results" / "figures"

train_path = processed_data_dir / "clean_train.parquet"
validation_path = processed_data_dir / "clean_validation.parquet"
test_path = processed_data_dir / "clean_test.parquet"

drive_base = Path("/content/drive/MyDrive/USW_Dissertation/Prompt-Injection-Detection-System-SHAP")
drive_models_dir = drive_base / "models"
drive_distilbert_dir = drive_models_dir / "distilbert"
drive_roberta_dir = drive_models_dir / "roberta"
drive_secbert_dir = drive_models_dir / "secbert"

print("paths defined")


# 9 check folders
folders_to_check = {
    "project root": project_root,
    "raw data": raw_data_dir,
    "processed data": processed_data_dir,
    "notebooks": notebooks_dir,
    "results tables": results_tables_dir,
    "results figures": results_figures_dir,
    "drive base": drive_base,
    "drive models": drive_models_dir,
    "drive distilbert": drive_distilbert_dir,
    "drive roberta": drive_roberta_dir,
    "drive secbert": drive_secbert_dir,
}

print("folder check")
for name, path in folders_to_check.items():
    print(name, "=", path.exists(), "|", path)


# 10 check dataset files
dataset_files_to_check = {
    "train": train_path,
    "validation": validation_path,
    "test": test_path,
}

print("dataset file check")
for name, path in dataset_files_to_check.items():
    print(name, "=", path.exists(), "|", path)


# 11 check colab environment
try:
    import google.colab
    running_in_colab = True
except ImportError:
    running_in_colab = False

print("environment check")
print("running in colab:", running_in_colab)
print("python:", sys.version.split()[0])
print("platform:", platform.platform())
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("sklearn:", sklearn.__version__)


# 12 final setup check
all_folders_exist = all(path.exists() for path in folders_to_check.values())
all_dataset_files_exist = all(path.exists() for path in dataset_files_to_check.values())

print("final check")
print("all folders exist:", all_folders_exist)
print("all dataset files exist:", all_dataset_files_exist)

if all_folders_exist and all_dataset_files_exist and running_in_colab:
    print("part 2 complete")
else:
    print("part 2 needs checking")